# Sensitivity analyses and narrative findings

Run this notebook in a Python environment with the required dependencies. By default it **runs the nine sensitivity analyses sequentially** from the frozen Stage 1 and Stage 4 inputs, verifies each output manifest, and writes a timestamped parent run plus a prose HTML report. No earlier sensitivity output folders are required.

The run can take about an hour because the training-size analysis refits models. If interrupted, set `RESUME_RUN_DIR` in the setup cell to the new `outputs/S1_04_sensitivity_...` folder and rerun; verified completed steps are reused. `RUN_MODE = "collect_latest"` is an optional shortcut only when all nine completed child outputs already exist. The report computes its numbers from the outputs of this run.

In [1]:
from pathlib import Path
import os, sys
from IPython.display import display
import pandas as pd

RUN_MODE = os.environ.get("REPLICATION_SENSITIVITY_MODE", "rerun_all")  # use "collect_latest" only with verified prior outputs
RESUME_RUN_DIR = os.environ.get("REPLICATION_SENSITIVITY_RESUME_DIR")  # or set a parent output-folder path

roots = [Path.cwd(), *Path.cwd().parents]
folder = next((candidate for root in roots for candidate in (
    root, root / "S1/sensitivity")
    if (candidate / "run_all_sensitivity.py").is_file()), None)
if folder is None:
    raise FileNotFoundError("Open this notebook from the replication workspace or sensitivity folder")
sys.path.insert(0, str(folder))
from run_all_sensitivity import run
print("Mode:", RUN_MODE, "| Source:", folder)

Mode: rerun_all | Source: /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/sensitivity


In [2]:
combined_run = run(mode=RUN_MODE, resume_dir=RESUME_RUN_DIR)
index = pd.read_csv(combined_run / "run_index.csv")
assert len(index) == 9 and index.analysis_status.str.startswith("complete").all()

saved_S1_S4: complete_descriptive_saved_result_sensitivity -> S1_S4_saved_sensitivity_20260929_210000_688594
SA01_adjustment: complete_exploratory_adjustment_refit -> S1_adjustment_without_PSYPROB_20260929_210005_969352
SA02_missingness: complete_descriptive_covariate_missingness -> S1_covariate_missingness_20260929_210557_980133
SA03_overlap: complete_frozen_policy_overlap_sensitivity -> S1_overlap_sensitivity_20260929_210601_499157
SA04_frequency: complete_descriptive_frequency_scenarios -> S1_outcome_observation_20260929_210608_634580
SA05_confounding: complete_exploratory_point_scenarios_not_calibrated -> S1_unmeasured_confounding_20260929_210734_607043
SA06_estimators: complete_descriptive_estimator_sensitivity -> S1_estimator_sensitivity_20260929_210739_635676
SA06b_nuisance: complete_exploratory_nuisance_refit -> S1_nuisance_refit_20260929_210741_508243
SA07_sample_size: complete_exploratory_sample_size_stability -> S1_sample_size_stability_20260929_210755_607496
Combined run: /

In [3]:
from pathlib import Path
from IPython.display import Markdown

runs = {row.step: Path(row.output_folder) for row in index.itertuples()}
def result(step, name):
    return pd.read_csv(runs[step] / name)
def pp(value):
    return f"{100 * value:+.2f} percentage points"
def count(value):
    return f"{int(value):,}"

primary = result("saved_S1_S4", "S1_completion/policy_branch_summary.csv").set_index("branch")
res = primary.loc["residential_eligible"]
out = primary.loc["outpatient_eligible"]
res_cc = primary.loc["residential_complete_case"]
out_cc = primary.loc["outpatient_complete_case"]
out_2023 = primary.loc["outpatient_temporal_2023"]
out_2023_fill = primary.loc["outpatient_temporal_mode_filled_2023"]

adjust = result("SA01_adjustment", "paired_refit_comparison.csv").set_index("branch")
selection = result("SA02_missingness", "state_year_treatment_selection.csv")
complete = selection.groupby("branch")[["eligible_n", "complete_case_n"]].sum()
missing = result("SA02_missingness", "covariate_missingness.csv")
missing_top = missing.sort_values("missing_fraction", ascending=False).groupby("branch").first()
overlap = result("SA03_overlap", "overlap_policy_sensitivity.csv")
res_support = overlap[(overlap.branch == "residential_eligible") & (overlap.scenario == "restrict_propensity_support") & (overlap.threshold == .05)].iloc[0]
out_support = overlap[(overlap.branch == "outpatient_eligible") & (overlap.scenario == "restrict_propensity_support") & (overlap.threshold == .05)].iloc[0]
out_clip = overlap[(overlap.branch == "outpatient_eligible") & (overlap.scenario == "clip_score_denominator")]

availability = result("SA04_frequency", "outcome_availability.csv").set_index(["setting", "treatment"])
frequency = result("SA04_frequency", "frequency_missing_outcome_scenarios.csv").groupby("setting")["arm1_minus_arm0_descriptive"].agg(["min", "max"])
confounding = result("SA05_confounding", "hypothetical_confounding_grid.csv")
def assumed(branch, contrast, strength):
    return confounding[(confounding.branch == branch) & (confounding.contrast == contrast) &
                       (confounding.cf_y_assumed == strength) &
                       (confounding.cf_d_assumed == strength)].iloc[0]
cf03_0 = assumed("outpatient_eligible", "tree_vs_always_0", .03)
cf03_1 = assumed("outpatient_eligible", "tree_vs_always_1", .03)
cf10_0 = assumed("outpatient_eligible", "tree_vs_always_0", .10)
cf10_1 = assumed("outpatient_eligible", "tree_vs_always_1", .10)
calibration = result("SA05_confounding", "PSYPROB_calibration_diagnostic.csv")
observed_omission = result("SA05_confounding", "observed_PSYPROB_omission.csv")
out_omission = observed_omission[(observed_omission.branch == "outpatient_eligible") &
                                (observed_omission.contrast == "tree_vs_always_0")].iloc[0]

estimators = result("SA06_estimators", "estimator_summary.csv")
aug_res = estimators[(estimators.branch == "residential") & (estimators.method == "overlap_augmented")].iloc[0]
aug_out = estimators[(estimators.branch == "outpatient") & (estimators.method == "overlap_augmented")].iloc[0]
nuisance = result("SA06b_nuisance", "nuisance_refit_policy_values.csv")
nuisance_out = nuisance[(nuisance.branch == "outpatient_eligible") &
                        (nuisance.nuisance_configuration != "original_nuisance")]
nuisance_res = nuisance[(nuisance.branch == "residential_eligible") &
                        (nuisance.nuisance_configuration != "original_nuisance")]
size = result("SA07_sample_size", "sample_size_summary.csv")
small_res = size[(size.branch == "residential") & (size.train_fraction == .25)].iloc[0]
small_out = size[(size.branch == "outpatient") & (size.train_fraction == .25)].iloc[0]
large = size[(size.train_fraction == .5) | (size.train_fraction == .75)]
s4 = result("saved_S1_S4", "S4_generator_evaluator/scenario_summary.csv")
self_removed = s4[s4.scenario == "exclude_generator_evaluator_overlap"].iloc[0]
omit_one = s4[s4.scenario == "omit_one_evaluator"]

sections = [f"""## Sensitivity results: recorded administrative completion

The primary comparisons are **long-term versus short-term residential care** and **intensive versus non-intensive outpatient care**. In the fixed held-out eligible samples, the residential tree assigns long-term care to {res.tree_treatment_1_share:.1%} of {count(res.evaluation_n)} episodes and its estimated value equals the best constant policy (gain {pp(res.tree_minus_best_constant)}). The outpatient tree assigns intensive care to {out.tree_treatment_1_share:.1%} of {count(out.evaluation_n)} episodes; its estimated gain over the best constant policy is {pp(out.tree_minus_best_constant)}. These are overlap-weighted observational estimates of **recorded administrative completion**, not clinical recovery rates or treatment recommendations.

The following sections explain the nine verified sensitivity analyses. A percentage-point difference is a difference between two proportions; these summaries are descriptive unless an analysis explicitly supplies valid inference.
""",
 f"""### Saved Stage 1 sample and Stage 4 evaluator checks

**What was checked.** The saved Stage 1 analysis compares the eligible completion policies with complete-case and 2023 outpatient variants, state and year omission checks, and 20 imputation draws. These variants answer different questions and sometimes use different populations. The saved Stage 4 analysis varies which model evaluations of the generated hypotheses are retained.

**What we found.** Residential policy gain remains {pp(res_cc.tree_minus_best_constant)} in the complete-case sample, although the held-out count changes from {count(res.evaluation_n)} to {count(res_cc.evaluation_n)}. Outpatient gain is {pp(out.tree_minus_best_constant)} in the eligible sample, {pp(out_cc.tree_minus_best_constant)} in the {count(out_cc.evaluation_n)}-episode complete-case sample, and {pp(out_2023.tree_minus_best_constant)} in the 2023 temporal analysis. The post-result mode-filled 2023 variant gives {pp(out_2023_fill.tree_minus_best_constant)}. The eligible outpatient leave-one-state-out gain ranges from {pp(out.omitted_state_min_gain_vs_always_0)} to {pp(out.omitted_state_max_gain_vs_always_0)}; one omitted state produces a negative gain in the complete-case variant.

For Stage 4, there are {int(s4.hypotheses.iloc[0])} unique hypotheses. Excluding a generating model's own evaluation changes the eligible fraction for {int(self_removed.hypotheses_with_changed_eligible_fraction)} hypotheses and the median novelty rating for {int(self_removed.hypotheses_with_changed_novelty_median)}. Omitting any one evaluator changes eligibility summaries for {int(omit_one.hypotheses_with_changed_eligible_fraction.min())} hypotheses and novelty medians for {int(omit_one.hypotheses_with_changed_novelty_median.min())}–{int(omit_one.hypotheses_with_changed_novelty_median.max())}.

**Interpretation.** The complete-case and 2023 numbers cannot be treated as direct changes on one fixed population. The 2023 mode-filled variant was chosen after viewing earlier results, and the 2023 state crosswalk remains unverified. Evaluator agreement or stability does not prove that a hypothesis is valid or novel.
""",
 f"""### Psychological-problem adjustment (SA01)

**What was checked.** The analysis removes the recorded psychological-problem covariate `PSYPROB`, refits the residential and outpatient policy analyses, and evaluates the alternatives on the same held-out episodes.

**What we found.** Policy assignments do not change in either setting (disagreement {adjust.loc['residential_eligible','policy_disagreement_fraction']:.1%} residential; {adjust.loc['outpatient_eligible','policy_disagreement_fraction']:.1%} outpatient). The residential gain remains {pp(adjust.loc['residential_eligible','alternative_tree_minus_best_constant'])}. Outpatient gain changes from {pp(adjust.loc['outpatient_eligible','baseline_tree_minus_best_constant'])} to {pp(adjust.loc['outpatient_eligible','alternative_tree_minus_best_constant'])}. Mean absolute fitted treatment-effect changes are {pp(adjust.loc['residential_eligible','mean_absolute_CATE_change'])} and {pp(adjust.loc['outpatient_eligible','mean_absolute_CATE_change'])}, respectively.

**Interpretation.** The policy conclusions are insensitive to this one measured adjustment choice in this refit. This does not assess an unmeasured confounder.
""",
 f"""### Missing baseline covariates (SA02)

**What was checked.** For the frozen eligible completion populations, the analysis counts records with complete modeling covariates by state, year, and treatment.

**What we found.** Residential complete cases number {count(complete.loc['residential','complete_case_n'])} of {count(complete.loc['residential','eligible_n'])} ({complete.loc['residential','complete_case_n']/complete.loc['residential','eligible_n']:.1%}). Outpatient complete cases number {count(complete.loc['outpatient','complete_case_n'])} of {count(complete.loc['outpatient','eligible_n'])} ({complete.loc['outpatient','complete_case_n']/complete.loc['outpatient','eligible_n']:.1%}). The most frequently missing recorded field is `{missing_top.loc['residential','field']}` in residential care ({missing_top.loc['residential','missing_fraction']:.1%}) and `{missing_top.loc['outpatient','field']}` in outpatient care ({missing_top.loc['outpatient','missing_fraction']:.1%}).

**Interpretation.** Complete-case estimates use a selected subset, especially for outpatient care. These counts do not measure selection into the eligible sample, missing discharge outcomes, or outcomes for excluded episodes.
""",
 f"""### Propensity overlap and support (SA03)

**What was checked.** The fitted policies and nuisance predictions stay fixed. One calculation clips score denominators; another excludes held-out episodes with propensity outside a specified support interval. Thresholds of 0.01, 0.025, and 0.05 are reported without choosing a threshold based on its result.

**What we found.** At the strictest support restriction (0.05), {count(res_support.n_excluded)} of {count(res.evaluation_n)} residential and {count(out_support.n_excluded)} of {count(out.evaluation_n)} outpatient held-out episodes are excluded. Residential tree gain remains {pp(res_support.tree_minus_best_constant)}. Outpatient gain changes from {pp(out.tree_minus_best_constant)} to {pp(out_support.tree_minus_best_constant)}. Across the denominator-clipping scenarios, the largest absolute change in outpatient policy gain is {100*out_clip.tree_gain_change_from_reference.abs().max():.4f} percentage points.

**Interpretation.** The reported point gain changes little under these specific calculations. A support restriction changes the target population; denominator clipping changes the estimating score and may introduce bias. This is not a validation of causal confidence intervals.
""",
 f"""### Missing secondary frequency outcomes (SA04)

**What was checked.** The analysis assigns hypothetical improvement probabilities of 0, 0.25, 0.50, 0.75, or 1 to unobserved frequency outcomes in each treatment arm, yielding 25 scenarios per setting.

**What we found.** Frequency reduction is observed for {availability.loc[('residential',0),'observation_fraction']:.1%} of short-term and {availability.loc[('residential',1),'observation_fraction']:.1%} of long-term residential episodes; the corresponding outpatient fractions are {availability.loc[('outpatient',0),'observation_fraction']:.1%} non-intensive and {availability.loc[('outpatient',1),'observation_fraction']:.1%} intensive. Across the 25 scenarios, the unadjusted long-minus-short residential contrast ranges from {pp(frequency.loc['residential','min'])} to {pp(frequency.loc['residential','max'])}; the intensive-minus-non-intensive outpatient contrast ranges from {pp(frequency.loc['outpatient','min'])} to {pp(frequency.loc['outpatient','max'])}.

**Interpretation.** Outpatient descriptive direction depends strongly on the assumed outcomes of missing records. These are unadjusted scenarios for a secondary outcome, not causal effect bounds or a sensitivity analysis of recorded completion.
""",
 f"""### Unmeasured-confounding scenarios (SA05)

**What was checked.** Fixed-policy point estimates are placed under a grid of assumed outcome and Riesz-strength parameters. Separately, removing observed `PSYPROB` from nuisance fits is used as a measured-variable stress test. The grid is hypothetical; it is not calibrated to an identified omitted variable.

**What we found.** For outpatient tree-versus-always-non-intensive and tree-versus-always-intensive comparisons, equal assumed strength parameters of 0.03 leave positive point-bound lower endpoints of {pp(cf03_0.point_bound_lower)} and {pp(cf03_1.point_bound_lower)}. At 0.10, both point-bound intervals contain zero ({bool(cf10_0.bound_includes_zero)} and {bool(cf10_1.bound_includes_zero)}). Omitting observed `PSYPROB` changes the outpatient tree-versus-always-non-intensive point contrast by {pp(out_omission.omit_minus_full)} on the fixed target. The proxy calibration check fails for {int((~calibration.proxy_check_pass.astype(bool)).sum())} of {len(calibration)} contrasts, and formal calibration is not validated for any contrast.

**Interpretation.** These point scenarios show how conclusions depend on stated assumptions; they are not confidence bounds, a measure of actual hidden bias, or proof that the policy is causally superior. The measured `PSYPROB` omission cannot calibrate the hypothetical grid.
""",
 f"""### Estimator choice across imputation draws (SA06)

**What was checked.** Overlap-weighted augmented estimation, normalized overlap inverse-probability weighting, and overlap outcome standardization are calculated on the same completed episodes and saved nuisance fits across 20 imputation draws per setting.

**What we found.** The augmented treatment contrast averages {pp(aug_res['mean'])} for residential and {pp(aug_out['mean'])} for outpatient care. Across the three estimators, residential means span {pp(estimators[estimators.branch=='residential']['mean'].min())} to {pp(estimators[estimators.branch=='residential']['mean'].max())}; outpatient means span {pp(estimators[estimators.branch=='outpatient']['mean'].min())} to {pp(estimators[estimators.branch=='outpatient']['mean'].max())}. The augmented estimate ranges across draws are {pp(aug_res.minimum)} to {pp(aug_res.maximum)} residential and {pp(aug_out.minimum)} to {pp(aug_out.maximum)} outpatient.

**Interpretation.** All three point contrasts have the same sign in these saved calculations, but they estimate contrasts using different formulas. The 20 completed datasets are dependent imputations, not independent replications, and these ranges are not confidence intervals.
""",
 f"""### Nuisance-learner choice (SA06b)

**What was checked.** The policy, held-out episodes, and original overlap target remain fixed while the outcome and propensity nuisance learners are replaced separately and together.

**What we found.** Outpatient policy gain over the best constant policy ranges from {pp(nuisance_out.tree_minus_best_constant.min())} to {pp(nuisance_out.tree_minus_best_constant.max())} across the three alternative configurations, versus {pp(out.tree_minus_best_constant)} originally. The residential gain remains {pp(nuisance_res.tree_minus_best_constant.min())} throughout. The outpatient always-intensive-minus-always-non-intensive contrast changes sign in the combined alternative fit, whose value is {pp(nuisance_out[nuisance_out.nuisance_configuration=='both_alternative'].always_1_minus_0.iloc[0])}.

**Interpretation.** Outpatient policy gain remains positive in these alternative point evaluations, while the average treatment contrast is sensitive near zero. This checks nuisance-learner dependence of a frozen policy, not stability of the policy-learning algorithm or statistical significance.
""",
 f"""### Training-sample size (SA07)

**What was checked.** The causal forests are refit five times at 25%, 50%, and 75% of their original training episodes. The original tree-build and held-out episodes stay fixed.

**What we found.** At 25% training size, maximum policy disagreement from the original is {small_res.max_policy_disagreement:.1%} for residential and {small_out.max_policy_disagreement:.1%} for outpatient care. Residential point gain ranges from {pp(small_res.min_gain)} to {pp(small_res.max_gain)}; outpatient point gain ranges from {pp(small_out.min_gain)} to {pp(small_out.max_gain)}. At 50% and 75% training size, maximum policy disagreement is {large.max_policy_disagreement.max():.1%} in the recorded refits.

**Interpretation.** Within these five seeds per fraction, the selected policies become more consistent at larger training fractions. The small number of refits and fixed held-out episodes do not establish general sampling stability or valid uncertainty intervals.
""",
 """### Overall interpretation

These analyses reveal where the reported results depend on sample completeness, propensity support, treatment-effect estimators, nuisance learners, assumed missing frequency outcomes, training size, and evaluator composition. They do not eliminate selection bias or unmeasured confounding, validate policy-gain confidence intervals, verify the unavailable 2023 state crosswalk, or establish the generated hypotheses as novel or true. The complete CSVs, child notebook reports, and manifests should be used for numerical audit."""]

display(Markdown("\n\n".join(sections)))


## Sensitivity results: recorded administrative completion

The primary comparisons are **long-term versus short-term residential care** and **intensive versus non-intensive outpatient care**. In the fixed held-out eligible samples, the residential tree assigns long-term care to 0.0% of 48,224 episodes and its estimated value equals the best constant policy (gain +0.00 percentage points). The outpatient tree assigns intensive care to 47.7% of 77,257 episodes; its estimated gain over the best constant policy is +2.49 percentage points. These are overlap-weighted observational estimates of **recorded administrative completion**, not clinical recovery rates or treatment recommendations.

The following sections explain the nine verified sensitivity analyses. A percentage-point difference is a difference between two proportions; these summaries are descriptive unless an analysis explicitly supplies valid inference.


### Saved Stage 1 sample and Stage 4 evaluator checks

**What was checked.** The saved Stage 1 analysis compares the eligible completion policies with complete-case and 2023 outpatient variants, state and year omission checks, and 20 imputation draws. These variants answer different questions and sometimes use different populations. The saved Stage 4 analysis varies which model evaluations of the generated hypotheses are retained.

**What we found.** Residential policy gain remains +0.00 percentage points in the complete-case sample, although the held-out count changes from 48,224 to 41,685. Outpatient gain is +2.49 percentage points in the eligible sample, +1.51 percentage points in the 48,946-episode complete-case sample, and +2.03 percentage points in the 2023 temporal analysis. The post-result mode-filled 2023 variant gives +2.30 percentage points. The eligible outpatient leave-one-state-out gain ranges from +1.55 percentage points to +3.29 percentage points; one omitted state produces a negative gain in the complete-case variant.

For Stage 4, there are 27 unique hypotheses. Excluding a generating model's own evaluation changes the eligible fraction for 13 hypotheses and the median novelty rating for 10. Omitting any one evaluator changes eligibility summaries for 21 hypotheses and novelty medians for 15–20.

**Interpretation.** The complete-case and 2023 numbers cannot be treated as direct changes on one fixed population. The 2023 mode-filled variant was chosen after viewing earlier results, and the 2023 state crosswalk remains unverified. Evaluator agreement or stability does not prove that a hypothesis is valid or novel.


### Psychological-problem adjustment (SA01)

**What was checked.** The analysis removes the recorded psychological-problem covariate `PSYPROB`, refits the residential and outpatient policy analyses, and evaluates the alternatives on the same held-out episodes.

**What we found.** Policy assignments do not change in either setting (disagreement 0.0% residential; 0.0% outpatient). The residential gain remains +0.00 percentage points. Outpatient gain changes from +2.49 percentage points to +2.52 percentage points. Mean absolute fitted treatment-effect changes are +1.66 percentage points and +1.68 percentage points, respectively.

**Interpretation.** The policy conclusions are insensitive to this one measured adjustment choice in this refit. This does not assess an unmeasured confounder.


### Missing baseline covariates (SA02)

**What was checked.** For the frozen eligible completion populations, the analysis counts records with complete modeling covariates by state, year, and treatment.

**What we found.** Residential complete cases number 208,421 of 241,118 (86.4%). Outpatient complete cases number 244,730 of 386,282 (63.4%). The most frequently missing recorded field is `PRIMINC` in residential care (3.3%) and `additional_substances_total` in outpatient care (15.3%).

**Interpretation.** Complete-case estimates use a selected subset, especially for outpatient care. These counts do not measure selection into the eligible sample, missing discharge outcomes, or outcomes for excluded episodes.


### Propensity overlap and support (SA03)

**What was checked.** The fitted policies and nuisance predictions stay fixed. One calculation clips score denominators; another excludes held-out episodes with propensity outside a specified support interval. Thresholds of 0.01, 0.025, and 0.05 are reported without choosing a threshold based on its result.

**What we found.** At the strictest support restriction (0.05), 4,428 of 48,224 residential and 267 of 77,257 outpatient held-out episodes are excluded. Residential tree gain remains +0.00 percentage points. Outpatient gain changes from +2.49 percentage points to +2.49 percentage points. Across the denominator-clipping scenarios, the largest absolute change in outpatient policy gain is 0.0002 percentage points.

**Interpretation.** The reported point gain changes little under these specific calculations. A support restriction changes the target population; denominator clipping changes the estimating score and may introduce bias. This is not a validation of causal confidence intervals.


### Missing secondary frequency outcomes (SA04)

**What was checked.** The analysis assigns hypothetical improvement probabilities of 0, 0.25, 0.50, 0.75, or 1 to unobserved frequency outcomes in each treatment arm, yielding 25 scenarios per setting.

**What we found.** Frequency reduction is observed for 97.8% of short-term and 96.7% of long-term residential episodes; the corresponding outpatient fractions are 72.7% non-intensive and 63.3% intensive. Across the 25 scenarios, the unadjusted long-minus-short residential contrast ranges from -6.85 percentage points to -1.36 percentage points; the intensive-minus-non-intensive outpatient contrast ranges from -30.97 percentage points to +33.01 percentage points.

**Interpretation.** Outpatient descriptive direction depends strongly on the assumed outcomes of missing records. These are unadjusted scenarios for a secondary outcome, not causal effect bounds or a sensitivity analysis of recorded completion.


### Unmeasured-confounding scenarios (SA05)

**What was checked.** Fixed-policy point estimates are placed under a grid of assumed outcome and Riesz-strength parameters. Separately, removing observed `PSYPROB` from nuisance fits is used as a measured-variable stress test. The grid is hypothetical; it is not calibrated to an identified omitted variable.

**What we found.** For outpatient tree-versus-always-non-intensive and tree-versus-always-intensive comparisons, equal assumed strength parameters of 0.03 leave positive point-bound lower endpoints of +0.51 percentage points and +0.98 percentage points. At 0.10, both point-bound intervals contain zero (True and True). Omitting observed `PSYPROB` changes the outpatient tree-versus-always-non-intensive point contrast by +0.02 percentage points on the fixed target. The proxy calibration check fails for 2 of 5 contrasts, and formal calibration is not validated for any contrast.

**Interpretation.** These point scenarios show how conclusions depend on stated assumptions; they are not confidence bounds, a measure of actual hidden bias, or proof that the policy is causally superior. The measured `PSYPROB` omission cannot calibrate the hypothetical grid.


### Estimator choice across imputation draws (SA06)

**What was checked.** Overlap-weighted augmented estimation, normalized overlap inverse-probability weighting, and overlap outcome standardization are calculated on the same completed episodes and saved nuisance fits across 20 imputation draws per setting.

**What we found.** The augmented treatment contrast averages -15.10 percentage points for residential and -1.85 percentage points for outpatient care. Across the three estimators, residential means span -15.52 percentage points to -15.02 percentage points; outpatient means span -2.14 percentage points to -1.20 percentage points. The augmented estimate ranges across draws are -15.12 percentage points to -15.08 percentage points residential and -1.87 percentage points to -1.84 percentage points outpatient.

**Interpretation.** All three point contrasts have the same sign in these saved calculations, but they estimate contrasts using different formulas. The 20 completed datasets are dependent imputations, not independent replications, and these ranges are not confidence intervals.


### Nuisance-learner choice (SA06b)

**What was checked.** The policy, held-out episodes, and original overlap target remain fixed while the outcome and propensity nuisance learners are replaced separately and together.

**What we found.** Outpatient policy gain over the best constant policy ranges from +2.29 percentage points to +2.67 percentage points across the three alternative configurations, versus +2.49 percentage points originally. The residential gain remains +0.00 percentage points throughout. The outpatient always-intensive-minus-always-non-intensive contrast changes sign in the combined alternative fit, whose value is +0.22 percentage points.

**Interpretation.** Outpatient policy gain remains positive in these alternative point evaluations, while the average treatment contrast is sensitive near zero. This checks nuisance-learner dependence of a frozen policy, not stability of the policy-learning algorithm or statistical significance.


### Training-sample size (SA07)

**What was checked.** The causal forests are refit five times at 25%, 50%, and 75% of their original training episodes. The original tree-build and held-out episodes stay fixed.

**What we found.** At 25% training size, maximum policy disagreement from the original is 3.5% for residential and 8.1% for outpatient care. Residential point gain ranges from -0.24 percentage points to +0.00 percentage points; outpatient point gain ranges from +2.54 percentage points to +2.73 percentage points. At 50% and 75% training size, maximum policy disagreement is 0.0% in the recorded refits.

**Interpretation.** Within these five seeds per fraction, the selected policies become more consistent at larger training fractions. The small number of refits and fixed held-out episodes do not establish general sampling stability or valid uncertainty intervals.


### Overall interpretation

These analyses reveal where the reported results depend on sample completeness, propensity support, treatment-effect estimators, nuisance learners, assumed missing frequency outcomes, training size, and evaluator composition. They do not eliminate selection bias or unmeasured confounding, validate policy-gain confidence intervals, verify the unavailable 2023 state crosswalk, or establish the generated hypotheses as novel or true. The complete CSVs, child notebook reports, and manifests should be used for numerical audit.